# Geodesic Walk — gradient steering along the curved manifold

**Direction:** `research/directions/geodesic-walk.md` (`[in-frame]`, sub-Q 3 editability).

**Question.** `findings/editability.md` localized the editing failure to **target unreachability under manifold constraint**: a single jump-then-project edit can't reach the target because the manifold curves away, and a global-PCA projection lands off the curved surface. The fix proposed here is to stop *jumping* and instead *walk*: take many small steps toward the target readout, **refitting the local tangent subspace at each step** and projecting onto it, climbing along the curved manifold. Clean binary: does readout RMSE → 0 as we iterate, or plateau?

**Central lesson driving this run (from the 2026-06-23 PCA run):** readout RMSE (decoded-position space) is the *convergence* metric, but a converged readout does **NOT** prove the model's *generated observation* moved. So we measure success in **BOTH spaces**: (1) readout RMSE for convergence, (2) observation space — does the generated 1D scan actually move toward the target object position, and is the phantom/ghost artifact gone?

This notebook is **self-contained** (cold-start bootstrap; does not rely on a live kernel). It mirrors `pca_component_position.ipynb` / `editability_structure.ipynb`. Both **printed metric tables** (for the agent) and **figures + PNG exports** (for Sevan) are deliverables.

Plan:
1. Setup — model, probe, global + bank-for-local subspaces, warm-up to edit, helpers.
2. One-shot baselines — pseudoinverse, global manifold, one-shot local (reproduce the editability table).
3. **Geodesic walk** — iterated small-step-toward-target + refit-local-tangent + project, logging readout RMSE / off-manifold residual / step size per iteration.
4. Convergence curves (decoded space).
5. **Observation-space outcome** — generated 1D scans + waterfalls, unsteered vs geodesic vs baselines; object-reaches-target + ghost-gone checks.
6. Verdict + PNG export.

---
## 1 — Setup: model, probe, global + local-bank subspaces, warm-up to edit

In [ ]:
import sys, os
sys.path.insert(0, "../../..")   # repo root -> import pim
sys.path.insert(0, "../..")      # notebooks/ -> helpers

from dataclasses import replace
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display

import pim.eval as eval
from pim.extractors import LinearExtractor, StateDefinition, identity_mse, hungarian_mse
from pim.editors import (
    probe_decomposition, inject_state,
    fit_state_subspace, project_to_subspace, offmanifold_residual,
    fit_local_subspace, manifold_steer, manifold_steer_local,
)
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset, make_test_loader

torch.manual_seed(0); np.random.seed(0)

CHECKPOINT_PATH = "../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
DATA_DIR        = "../../../datasets/4_fixed_refl_inview"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE      = 512
NUM_WORKERS     = 6

N_OBJ           = 2
USE_HUNGARIAN   = False     # fixed reflectivities -> identity matching
SUBSPACE_VAR    = 0.90      # variance kept by the GLOBAL state-manifold PCA
LOCAL_K         = 512       # nearest neighbours per local tangent patch
LOCAL_VAR       = 0.90      # variance kept WITHIN a local patch
LOCAL_BANK_SIZE = 50_000    # visited-state bank subsample for the kNN

# --- experiment scale (START SMALL per the brief: N<=64, K<=30) ---
N_GEO           = 64        # edit samples walked
N_ROLLOUT       = 15        # post-edit rollout length
K_ITERS         = 30        # geodesic-walk iterations
POCS_ITERS      = 50        # edit<->project alternations for the one-shot baselines

os.makedirs("/tmp/geodesic", exist_ok=True)

model, ckpt_info = load_checkpoint(CHECKPOINT_PATH, device=DEVICE)
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
test_loader = make_test_loader(test, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

# Teacher-force the test set -> bank of visited hidden states.
preds_tf, states_tf = eval.teacher_force(model, test_loader, device=DEVICE)

# Linear position probe.
state_def = StateDefinition(name="positions", state_shape=(N_OBJ, 2),
                            extract_fn=lambda b: b["positions"])
env_states_tf = test.positions[:, :-1, :N_OBJ, :]
vis_mask_tf   = test.is_visible[:, :-1, :N_OBJ].all(axis=2)
loss_fn       = hungarian_mse if USE_HUNGARIAN else identity_mse

linear = LinearExtractor(model.hidden_size, state_def, use_lstsq=True)
train_mse = linear.fit(states_tf, env_states_tf, mask=vis_mask_tf, loss_fn=loss_fn, device=DEVICE)
linear = linear.to(DEVICE).eval()

print(f"Model : {ckpt_info.run_name} (epoch {ckpt_info.epoch}, val_loss={ckpt_info.val_loss:.5f})")
print(f"Hidden: {model.hidden_size}  states_tf={states_tf.shape}")
print(f"Probe : linear position, train MSE={train_mse:.6f}  device={DEVICE}")

In [ ]:
# Global manifold subspace (on device) + on-device bank for local tangent fits.
subspace = fit_state_subspace(states_tf, var_threshold=SUBSPACE_VAR)
subspace_dev = replace(subspace,
    mean=subspace.mean.to(DEVICE), basis=subspace.basis.to(DEVICE),
    explained_variance_ratio=subspace.explained_variance_ratio.to(DEVICE))

_bank_all = states_tf.reshape(-1, model.hidden_size)
_sub = np.random.RandomState(0).choice(
    _bank_all.shape[0], size=min(LOCAL_BANK_SIZE, _bank_all.shape[0]), replace=False)
bank_dev = torch.from_numpy(_bank_all[_sub]).float().to(DEVICE)

# Off-manifold residual scale of REAL states (the on-manifold reference, global + local).
real_res_global = float(offmanifold_residual(
    torch.from_numpy(_bank_all[:5000]).float().to(DEVICE), subspace_dev).mean())
def _local_resid(h, n_probe=100):
    """Per-sample LOCAL-tangent off-manifold residual — the honest, curvature-aware detector."""
    res = []
    for i in range(min(n_probe, h.shape[0])):
        sub = fit_local_subspace(bank_dev, h[i], k_neighbors=LOCAL_K,
                                 var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)
        res.append(float(offmanifold_residual(h[i:i+1], sub).mean()))
    return float(np.mean(res))
real_res_local = _local_resid(torch.from_numpy(_bank_all[_sub[:200]]).float().to(DEVICE))
print(f"global subspace: kept {subspace.n_components}/{subspace.hidden_size} "
      f"({subspace.total_explained:.4f} var)")
print(f"REAL-state off-manifold residual:  global={real_res_global:.4f}  local={real_res_local:.4f}")

# Warm up to the edit frame -> base hidden states we will edit/walk from.
N = min(N_GEO, edits.n_samples)
warm = eval.warm_up_to_edit(model, edits.obs[:N], edits.edit_frame,
                            n_viz=N, n_ctx_show=8, device=DEVICE)
h_base = warm.h_at_edit[:N]                                   # (N, H) cold-start point

# Target readout = post-edit (teleported) GT positions at the edit frame.
targets = edits.positions[:N, edits.edit_frame, :N_OBJ, :].reshape(N, N_OBJ * 2)

# Probe decomposition (on device).
A, b, A_pinv = probe_decomposition(linear)
h0  = torch.from_numpy(h_base).float().to(DEVICE)
tgt = torch.from_numpy(targets).float().to(DEVICE)
edit_fn = lambda h, t: inject_state(h, t, A, A_pinv, b)

def readout(h):                          # (.,H) -> (.,N_OBJ*2)
    return h @ A.T + b
def readout_rmse(h, t=tgt):
    return float((readout(h) - t).pow(2).mean().sqrt())
def resid_global(h):
    return float(offmanifold_residual(h, subspace_dev).mean())

@torch.no_grad()
def rollout_from_flat(h_array, n_rollout):
    """Roll out from each flat state; step 0 = decode (no advance)."""
    obs_all, h_all = [], []
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o, hs = _rollout(model, h, n_rollout)
        obs_all.append(o); h_all.append(hs)
    return np.stack(obs_all), np.stack(h_all)

@torch.no_grad()
def decode_pos(h_array):
    t = torch.as_tensor(h_array, dtype=torch.float32, device=DEVICE)
    return linear(t).cpu().numpy()       # (...,N_OBJ,2)

print(f"\nh_base={h_base.shape}  edit_frame={edits.edit_frame}  N={N}  K_ITERS={K_ITERS}")
print(f"cold-start readout RMSE (unsteered vs target): {readout_rmse(h0):.4f}")

---
## 2 — One-shot baselines (reproduce the editability table)

The three existing one-shot editors from `findings/editability.md`:
- **pseudoinv** — `inject_state` (min-norm; off-manifold OK).
- **manifold** — `manifold_steer` against the *global* PCA subspace (POCS).
- **one-shot local** — `manifold_steer_local`: fit ONE local tangent subspace at the cold-start point and run POCS against it (no refit as the state moves).

These are the bars the geodesic walk must beat. Each gives `readout RMSE / global resid / local resid`. High manifold/local readout RMSE = the target is unreachable in that fixed neighbourhood (the curvature barrier).

In [ ]:
h_pinv     = inject_state(h0, tgt, A, A_pinv, b)                                   # off-manifold
h_manifold = manifold_steer(h0, tgt, edit_fn, subspace_dev, n_iters=POCS_ITERS)    # global manifold
h_local    = manifold_steer_local(h0, tgt, edit_fn, bank_dev,                      # one-shot local
                                  k_neighbors=LOCAL_K, n_iters=POCS_ITERS,
                                  var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)

print(f"{'edit':14s} {'readout RMSE':>13s} {'global resid':>13s} {'local resid':>12s}")
baseline_states = {"unsteered": h0, "pseudoinv": h_pinv,
                   "manifold": h_manifold, "one-shot local": h_local}
baseline_table = {}
for name, h in baseline_states.items():
    r_rmse, r_g, r_l = readout_rmse(h), resid_global(h), _local_resid(h)
    baseline_table[name] = (r_rmse, r_g, r_l)
    print(f"{name:14s} {r_rmse:13.4f} {r_g:13.4f} {r_l:12.4f}")
print(f"{'real states':14s} {'—':>13s} {real_res_global:13.4f} {real_res_local:12.4f}")
print("\nlocal resid is the honest off-manifold detector; high manifold/one-shot-local")
print("readout RMSE => target unreachable in a FIXED neighbourhood (the curvature barrier).")

---
## 3 — The geodesic walk

Iterate, starting from `h_at_edit`, for K steps. At each step:

1. **Small step toward target** — a *fractional* pseudoinverse nudge. `inject_state(h, target)` lands exactly on the readout constraint in one jump; we take a fraction `step_frac` of the way there: `h_step = h + step_frac · (inject_state(h, target) − h)`. (This is the "small linear pseudoinverse nudge" option in the brief. A gradient-step variant on the same loss is available via `gradient_steer`; the linear nudge is the cleaner reachability probe and is what we report.)
2. **Refit a FRESH local tangent subspace** around the *new* point (`fit_local_subspace`) and **project** onto it. This is the difference from one-shot local: the patch is re-localized every iteration, so the walk can follow the manifold's curvature.
3. **Log** readout RMSE, local off-manifold residual, and step size `‖Δh‖` each iteration.

This is `manifold_steer`'s edit↦project alternation, but with the projector **refit per step** — i.e. POCS where the convex set (local tangent plane) is re-estimated as the iterate moves. Reusing the primitives: the per-step edit is the fractional `inject_state`; the per-step projector is `project_to_subspace` onto a freshly `fit_local_subspace`'d patch.

`STEP_FRAC` controls aggressiveness. Smaller = more faithful geodesic (each projection is onto a near-valid tangent), at more iterations.

In [ ]:
from tqdm.auto import tqdm

STEP_FRAC = 0.34      # fraction of the full pseudoinverse jump taken per iteration

@torch.no_grad()
def geodesic_walk(h_start, target, *, k_iters=K_ITERS, step_frac=STEP_FRAC,
                  k_neighbors=LOCAL_K, var_threshold=LOCAL_VAR,
                  bank=bank_dev, bank_size=LOCAL_BANK_SIZE, log_residual=True):
    """Iterated small-step-toward-target + refit-local-tangent + project, per sample.

    Returns:
      h_final : (N, H) walked states
      log     : dict of (N, k_iters+1) arrays — 'rmse' (per-sample readout RMSE),
                'resid_local' (per-sample local off-manifold residual), 'step' (||Δh||, k_iters).
    """
    Nn = h_start.shape[0]
    h_out = torch.empty_like(h_start)
    rmse_log  = np.zeros((Nn, k_iters + 1))
    resid_log = np.full((Nn, k_iters + 1), np.nan)
    step_log  = np.zeros((Nn, k_iters))
    for i in tqdm(range(Nn), desc="geodesic walk", leave=False):
        h = h_start[i:i+1]                                  # (1, H)
        t = target[i:i+1]
        rmse_log[i, 0] = float((readout(h) - t).pow(2).mean().sqrt())
        if log_residual:
            sub0 = fit_local_subspace(bank, h[0], k_neighbors=k_neighbors,
                                      var_threshold=var_threshold, bank_size=bank_size)
            resid_log[i, 0] = float(offmanifold_residual(h, sub0).mean())
        for k in range(k_iters):
            h_full = inject_state(h, t, A, A_pinv, b)        # full jump onto readout constraint
            h_step = h + step_frac * (h_full - h)            # SMALL step toward target
            sub = fit_local_subspace(bank, h_step[0], k_neighbors=k_neighbors,   # FRESH local patch
                                     var_threshold=var_threshold, bank_size=bank_size)
            h_proj = project_to_subspace(h_step, sub)        # project onto re-localized tangent
            step_log[i, k] = float((h_proj - h).norm())
            h = h_proj
            rmse_log[i, k + 1] = float((readout(h) - t).pow(2).mean().sqrt())
            if log_residual:
                resid_log[i, k + 1] = float(offmanifold_residual(h, sub).mean())
        h_out[i] = h[0]
    return h_out, {"rmse": rmse_log, "resid_local": resid_log, "step": step_log}

h_geo, geo_log = geodesic_walk(h0, tgt)

# Summary table (means over samples), every few iterations.
rmse_m  = geo_log["rmse"].mean(0)
resid_m = geo_log["resid_local"].mean(0)
step_m  = geo_log["step"].mean(0)
print(f"STEP_FRAC={STEP_FRAC}  K_ITERS={K_ITERS}  N={N}")
print(f"{'iter':>5} {'readout RMSE':>13} {'local resid':>12} {'step ||dh||':>12}")
its = sorted(set([0, 1, 2, 5, 10, 15, 20, 25, K_ITERS]))
its = [k for k in its if k <= K_ITERS]
for k in its:
    s = f"{step_m[k-1]:12.4f}" if k >= 1 else f"{'—':>12}"
    print(f"{k:>5} {rmse_m[k]:13.4f} {resid_m[k]:12.4f} {s}")
print(f"\nreal-state local resid reference = {real_res_local:.4f}")
print(f"final geodesic readout RMSE = {rmse_m[-1]:.4f}  (cold start = {rmse_m[0]:.4f})")

---
## 4 — Convergence curves (decoded space) + geodesic vs baseline readout

Does readout RMSE → 0 (barrier traversable) or plateau (different attractor)? And does the local off-manifold residual stay ≈ the real-state reference throughout (i.e. the walk stays on-manifold)?

In [ ]:
# Combined readout table: geodesic final vs the one-shot baselines.
geo_rmse_final = readout_rmse(h_geo)
geo_resid_g    = resid_global(h_geo)
geo_resid_l    = _local_resid(h_geo)
print("=== READOUT-SPACE COMPARISON (geodesic vs one-shot baselines) ===")
print(f"{'edit':16s} {'readout RMSE':>13s} {'global resid':>13s} {'local resid':>12s}")
for name, (r, g, l) in baseline_table.items():
    print(f"{name:16s} {r:13.4f} {g:13.4f} {l:12.4f}")
print(f"{'GEODESIC walk':16s} {geo_rmse_final:13.4f} {geo_resid_g:13.4f} {geo_resid_l:12.4f}")
print(f"{'real states':16s} {'—':>13s} {real_res_global:13.4f} {real_res_local:12.4f}")
print(f"\non-manifold check: geodesic local resid {geo_resid_l:.3f} vs real {real_res_local:.3f} "
      f"(ratio {geo_resid_l/real_res_local:.2f})")

iters = np.arange(K_ITERS + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) readout RMSE vs iteration — per-sample faint, mean bold; baselines as hlines.
ax = axes[0]
for i in range(N):
    ax.plot(iters, geo_log["rmse"][i], color="0.8", lw=0.5, alpha=0.5, zorder=1)
ax.plot(iters, rmse_m, color="#0072B2", lw=2.5, marker="o", ms=3, label="geodesic (mean)", zorder=3)
for name, c, ls in [("pseudoinv", "#D55E00", ":"), ("manifold", "#009E73", "--"),
                    ("one-shot local", "#CC79A7", "-.")]:
    ax.axhline(baseline_table[name][0], color=c, ls=ls, lw=1.5, label=f"{name} (one-shot)")
ax.set_xlabel("iteration"); ax.set_ylabel("readout RMSE (decoded space)")
ax.set_title("(a) convergence: readout RMSE -> 0 or plateau?")
ax.grid(alpha=0.3); ax.legend(fontsize=8)

# (b) local off-manifold residual vs iteration — does the walk stay on-manifold?
ax = axes[1]
ax.plot(iters, resid_m, color="#0072B2", lw=2.5, marker="o", ms=3, label="geodesic local resid (mean)")
ax.axhline(real_res_local, color="0.4", ls="--", lw=1.5, label=f"real-state local resid={real_res_local:.3f}")
ax.set_xlabel("iteration"); ax.set_ylabel("local off-manifold residual")
ax.set_title("(b) on-manifold? (residual vs real reference)")
ax.grid(alpha=0.3); ax.legend(fontsize=8)

# (c) step size vs iteration — is it converging (steps -> 0) or wandering?
ax = axes[2]
ax.plot(np.arange(1, K_ITERS + 1), step_m, color="#0072B2", lw=2.0, marker="o", ms=3)
ax.set_xlabel("iteration"); ax.set_ylabel("mean step size ||dh||")
ax.set_title("(c) step size per iteration"); ax.grid(alpha=0.3)

fig.suptitle("Geodesic walk: convergence in decoded space", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig("/tmp/geodesic/1_convergence.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved /tmp/geodesic/1_convergence.png")

---
## 5 — Observation-space outcome (THE central check)

A converged readout RMSE does **not** prove the model's generated observation moved. So we roll out each edited state and ask, in the space the model actually generates:

- **Does the object reach the target in obs space?** We render the *target* scene (post-edit GT positions) and measure how close the **geodesic-generated** obs is to that target render, vs the unsteered and baseline edits. Closer to the target render = the edit landed in obs space.
- **Is the ghost/phantom artifact gone?** The ghost is intensity left at the object's *original* (pre-edit) location after a successful edit (incomplete identity displacement). We measure residual intensity at the pre-edit object location relative to the unsteered (no-edit) scan there: ghost ratio ≈ 1 means the old object is still fully present; ≈ 0 means cleanly removed.

We render targets via `pim.simulator.renderer` from the GT post-edit positions (a clean reference the model should match if the edit fully worked). `obs_noise_std=0` so we compare to denoised references.

In [ ]:
# Roll out every variant from its edited state (step 0 = decode, no advance).
variant_h = {
    "unsteered":      h0,
    "pseudoinv":      h_pinv,
    "manifold":       h_manifold,
    "one-shot local": h_local,
    "geodesic":       h_geo,
}
roll_obs, roll_hs = {}, {}
for name, h in variant_h.items():
    o, hs = rollout_from_flat(h.detach().cpu().numpy(), N_ROLLOUT)
    roll_obs[name] = o          # (N, N_ROLLOUT, R)
    roll_hs[name]  = hs
OBS_RES = roll_obs["unsteered"].shape[-1]
print("rollouts:", {k: v.shape for k, v in roll_obs.items()})

# Render the TARGET scene from the post-edit GT positions (the obs the edit should produce).
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
sim = test.config["dataset"]["sim"]
def make_cfg(n_frames):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"],
                     x_far=sim["x_far"], n_objects=N_OBJ, radius=sim["radius"],
                     n_frames=n_frames, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"],
                     fixed_reflectivities=True, obs_noise_std=0.0, boundary="open",
                     always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], dtype=np.float32)   # obj0=min(dim), obj1=max(bright)
RAD  = np.array([sim["radius"]] * N_OBJ, dtype=np.float32)
COL  = np.tile(np.array([[1, 1, 1]], dtype=np.float32), (N_OBJ, 1))

# Target render: GT post-edit positions at the edit frame, static (step-0 reference for each rollout step).
tgt_pos = edits.positions[:N, edits.edit_frame, :N_OBJ, :].astype(np.float32)   # (N, N_OBJ, 2)
pre_pos = edits.positions[:N, edits.edit_frame - 1, :N_OBJ, :].astype(np.float32)
cfg1 = make_cfg(1)
tgt_render_id  = np.zeros((N, OBS_RES), dtype=np.int64)
tgt_render_int = np.zeros((N, OBS_RES), dtype=np.float32)
pre_render_id  = np.zeros((N, OBS_RES), dtype=np.int64)
for i in range(N):
    sc = Scene(positions=tgt_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32),
               radii=RAD, colors=COL, reflectivities=REFL, config=cfg1)
    _, rid, rint = render_scene(sc)
    tgt_render_id[i], tgt_render_int[i] = rid[0], rint[0]
    sc_pre = Scene(positions=pre_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32),
                   radii=RAD, colors=COL, reflectivities=REFL, config=cfg1)
    _, rid_pre, _ = render_scene(sc_pre)
    pre_render_id[i] = rid_pre[0]
print("target-render scenes built:", tgt_render_int.shape)
print("edit_object distribution:", np.bincount(edits.edit_object[:N]))

In [ ]:
# ---- Observation-space metrics ----
obs_u = roll_obs["unsteered"]                       # (N, N_ROLLOUT, R)
edit_obj = edits.edit_object[:N]                    # which object was teleported

def rms(a, b):  return float(np.sqrt(((a - b) ** 2).mean()))

# (1) distance to TARGET render (step 0 = the direct edit; lower = object reached target in obs space).
#     reference scale: how far the unsteered (no-edit) scan is from the target render.
def dist_to_target_render(obs, step=0):
    return rms(obs[:, step, :], tgt_render_int)     # compare generated scan vs target render
unsteered_to_target = dist_to_target_render(obs_u)
# full-change reference: target render vs the unsteered scan = the gap a perfect edit must close.

# (2) obs CHANGE vs unsteered (did the edit move the output at all?).
def obs_change(obs, step=0):
    return rms(obs[:, step, :], obs_u[:, step, :])

# (3) GHOST ratio: residual intensity at the PRE-EDIT location of the teleported object,
#     in the generated scan, relative to the unsteered scan there. ~1 = old object still
#     fully present (ghost); ~0 = cleanly removed. Only over rays that hit the edited object
#     in the PRE-edit render AND are NOT covered by the target object render (avoid double count).
ghost_mask = np.zeros((N, OBS_RES), dtype=bool)
for i in range(N):
    m = (pre_render_id[i] == edit_obj[i]) & (tgt_render_id[i] != edit_obj[i])
    ghost_mask[i] = m
ghost_denom = ghost_mask.sum()
def ghost_ratio(obs, step=0):
    """mean intensity on ghost rays (generated) / mean intensity on ghost rays (unsteered)."""
    if ghost_denom == 0:
        return np.nan
    g_obs = obs[:, step, :][ghost_mask].mean()
    g_uns = obs_u[:, step, :][ghost_mask].mean()
    return float(g_obs / g_uns) if g_uns > 1e-6 else np.nan

print(f"ghost-region rays available: {int(ghost_denom)}  "
      f"(mean unsteered intensity there = {obs_u[:, 0, :][ghost_mask].mean():.3f})")
print(f"reference gap: unsteered scan vs target render (step0) = {unsteered_to_target:.4f} "
      f"(a perfect edit closes this to ~render noise floor)\n")

print("=== OBSERVATION-SPACE TABLE (step 0 = direct edit) ===")
print(f"{'variant':16s} {'->target render':>15s} {'obs change':>11s} {'ghost ratio':>12s}")
obs_table = {}
for name, obs in roll_obs.items():
    d_t, d_c, g = dist_to_target_render(obs), obs_change(obs), ghost_ratio(obs)
    obs_table[name] = (d_t, d_c, g)
    print(f"{name:16s} {d_t:15.4f} {d_c:11.4f} {g:12.3f}")
print(f"\nlower '->target render' = closer to the obs a perfect edit makes; "
      f"unsteered baseline={unsteered_to_target:.4f}.")
print("ghost ratio ~0 = old object removed (no phantom); ~1 = still fully present.")

# Also a step-resolved version (does the obs-space effect persist through the rollout?).
steps = np.arange(N_ROLLOUT)
to_tgt_step = {n: np.array([dist_to_target_render(o, s) for s in steps]) for n, o in roll_obs.items()}
chg_step    = {n: np.array([obs_change(o, s) for s in steps]) for n, o in roll_obs.items() if n != "unsteered"}
ghost_step  = {n: np.array([ghost_ratio(o, s) for s in steps]) for n, o in roll_obs.items()}
print("\nmean '->target render' over rollout: " +
      "  ".join(f"{n}={v.mean():.3f}" for n, v in to_tgt_step.items()))

In [ ]:
# Decoded-position distance to target (step 0) — ties readout space to the obs-space table above.
# Per-OBJECT, so we can see whether the edited object's DECODED position reaches target while obs lags.
tgt_pos_np = targets.reshape(N, N_OBJ, 2)
def dec_dist_per_obj(hs):
    pos0 = decode_pos(hs[:, 0])                                  # (N, N_OBJ, 2)
    return np.sqrt(((pos0 - tgt_pos_np) ** 2).sum(-1)).mean(0)   # (N_OBJ,)
print("=== DECODED-POSITION distance to target (step 0), per object ===")
print(f"{'variant':16s} " + " ".join(f"{'obj'+str(o):>9s}" for o in range(N_OBJ)) + f" {'mean':>9s}")
for name, hs in roll_hs.items():
    d = dec_dist_per_obj(hs)
    print(f"{name:16s} " + " ".join(f"{x:9.4f}" for x in d) + f" {d.mean():9.4f}")
print("\nIf geodesic decoded-dist -> ~0 but its '->target render' stays high, the probe reports")
print("motion the model does not generate (the BOTH-SPACES disagreement the brief warns about).")

In [ ]:
# Step-resolved obs-space figure: does the obs effect reach target & persist? Is the ghost gone?
COL = {"unsteered": "0.5", "pseudoinv": "#D55E00", "manifold": "#009E73",
       "one-shot local": "#CC79A7", "geodesic": "#0072B2"}
MK  = {"unsteered": None, "pseudoinv": "v", "manifold": "s", "one-shot local": "d", "geodesic": "o"}

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))

ax = axes[0]
for n, v in to_tgt_step.items():
    ax.plot(steps, v, color=COL[n], marker=MK[n], ms=3,
            lw=2.2 if n == "geodesic" else 1.4, label=n)
ax.set_xlabel("rollout step"); ax.set_ylabel("RMS(generated obs, TARGET render)")
ax.set_title("(a) does the obs reach the target?\n(lower = object at target in obs space)")
ax.grid(alpha=0.3); ax.legend(fontsize=8)

ax = axes[1]
for n, v in chg_step.items():
    ax.plot(steps, v, color=COL[n], marker=MK[n], ms=3,
            lw=2.2 if n == "geodesic" else 1.4, label=n)
ax.set_xlabel("rollout step"); ax.set_ylabel("RMS obs change vs unsteered")
ax.set_title("(b) did the edit move the output?"); ax.grid(alpha=0.3); ax.legend(fontsize=8)

ax = axes[2]
for n, v in ghost_step.items():
    ax.plot(steps, v, color=COL[n], marker=MK[n], ms=3,
            lw=2.2 if n == "geodesic" else 1.4, label=n)
ax.axhline(1.0, color="0.7", ls="--", lw=1, label="old object fully present")
ax.axhline(0.0, color="0.7", ls=":", lw=1, label="old object removed")
ax.set_xlabel("rollout step"); ax.set_ylabel("ghost ratio (pre-edit location)")
ax.set_title("(c) is the phantom/ghost gone?"); ax.grid(alpha=0.3); ax.legend(fontsize=7)

fig.suptitle("Observation-space outcome: geodesic vs baselines", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig("/tmp/geodesic/2_obs_space_metrics.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved /tmp/geodesic/2_obs_space_metrics.png")

### 5b — Generated 1D scans + waterfalls (the human's judging surface)

For representative samples (largest GT teleport, so the target shift is big and legible): overlay the generated 1D intensity scan at the direct-edit step for unsteered / geodesic / each baseline, against the **target render** (black dashed) and the **pre-edit object location** (ghost zone, shaded). Then the per-variant waterfalls with the target object centroid (green) and pre-edit/ghost centroid (red) overlaid.

In [ ]:
# Pick samples with the largest GT teleport AND a resolvable ghost zone (so the edit is legible).
teleport = np.linalg.norm(tgt_pos - pre_pos, axis=-1)[np.arange(N), edit_obj]   # edited-obj shift
has_ghost = ghost_mask.sum(1) >= 3
score = teleport * has_ghost
SAMPLES = list(np.argsort(score)[::-1][:3])
print("representative samples (big teleport + ghost zone):", SAMPLES,
      "teleport=", [round(float(teleport[s]), 2) for s in SAMPLES])

rays = np.arange(OBS_RES)
SCAN_STEP = 0     # direct-edit step
order = ["unsteered", "pseudoinv", "manifold", "one-shot local", "geodesic"]

fig, axes = plt.subplots(len(SAMPLES), 1, figsize=(11, 3.0 * len(SAMPLES)), squeeze=False)
for r, smp in enumerate(SAMPLES):
    ax = axes[r][0]
    # target render (what a perfect edit should produce) + ghost zone shading
    ax.plot(rays, tgt_render_int[smp], color="k", ls="--", lw=1.6, label="TARGET render", zorder=5)
    gz = np.where(ghost_mask[smp])[0]
    if gz.size:
        ax.axvspan(gz.min() - 0.5, gz.max() + 0.5, color="red", alpha=0.10, zorder=0,
                   label="pre-edit (ghost) zone")
    for n in order:
        ax.plot(rays, roll_obs[n][smp, SCAN_STEP], color=COL[n],
                lw=2.4 if n == "geodesic" else 1.4, alpha=0.95 if n == "geodesic" else 0.8,
                label=n, zorder=4 if n == "geodesic" else 2)
    ax.set_title(f"sample {smp}  (edited obj {edit_obj[smp]}, teleport={teleport[smp]:.2f})")
    ax.set_xlabel("ray index"); ax.set_ylabel("intensity"); ax.set_ylim(-0.02, 1.05)
    ax.grid(alpha=0.25); ax.legend(fontsize=7, ncol=3, loc="upper right")
fig.suptitle(f"Generated 1D scans at direct-edit step (step {SCAN_STEP}) — does geodesic match the TARGET render?",
             y=1.005, fontsize=12)
fig.tight_layout()
fig.savefig("/tmp/geodesic/3_scans.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved /tmp/geodesic/3_scans.png")

In [ ]:
# Waterfalls: rows = samples, cols = variants. Grayscale = generated intensity.
# Green vline = target-render centroid of edited obj (where it SHOULD be); red = pre-edit (ghost) centroid.
def centroid(mask_row):
    idx = np.where(mask_row)[0]
    return idx.mean() if idx.size else np.nan

fig, axes = plt.subplots(len(SAMPLES), len(order),
                         figsize=(2.7 * len(order), 3.0 * len(SAMPLES)), squeeze=False)
for r, smp in enumerate(SAMPLES):
    tgt_cx = centroid(tgt_render_id[smp] == edit_obj[smp])
    pre_cx = centroid(pre_render_id[smp] == edit_obj[smp])
    for c, n in enumerate(order):
        ax = axes[r][c]
        ax.imshow(roll_obs[n][smp], aspect="auto", origin="upper", cmap="gray",
                  vmin=0, vmax=1, interpolation="nearest")
        if not np.isnan(tgt_cx):
            ax.axvline(tgt_cx, color="#00E676", ls="-", lw=1.4, alpha=0.9)
        if not np.isnan(pre_cx):
            ax.axvline(pre_cx, color="#FF5252", ls="--", lw=1.4, alpha=0.9)
        if r == 0:
            ax.set_title(n, fontsize=10)
        if c == 0:
            ax.set_ylabel(f"smp {smp}\nframe", fontsize=9)
        ax.set_xlabel("ray", fontsize=8)
axes[0][0].plot([], [], color="#00E676", lw=2, label="target loc (edited obj)")
axes[0][0].plot([], [], color="#FF5252", ls="--", lw=2, label="pre-edit (ghost) loc")
axes[0][0].legend(loc="upper right", fontsize=6)
fig.suptitle("Generated waterfalls: green = where edited obj SHOULD be, red = ghost zone\n"
             "(good edit: bright streak at green, nothing at red)", y=1.015, fontsize=12)
fig.tight_layout()
fig.savefig("/tmp/geodesic/4_waterfalls.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved /tmp/geodesic/4_waterfalls.png")

---
## 6 — Verdict (auto-summarized)

Reads the computed numbers and prints the four answers the brief asks for. See `research/scratch/2026-06-23-geodesic-walk.md` for the written note. **Not promoted to findings.**

In [ ]:
print("=" * 70)
print("GEODESIC WALK — AUTO VERDICT  (N={}, K={}, step_frac={})".format(N, K_ITERS, STEP_FRAC))
print("=" * 70)

# (a) convergence in decoded/readout space
cold, final = rmse_m[0], rmse_m[-1]
tail = rmse_m[-5:]
plateaued = abs(tail[-1] - tail[0]) < 0.02 and final > 0.05
verdict_a = "CONVERGES to ~0" if final < 0.05 else (
    f"PLATEAUS at RMSE={final:.3f}" if plateaued else f"still decreasing, RMSE={final:.3f}")
print(f"\n(a) CONVERGENCE (readout/decoded space): {verdict_a}")
print(f"    cold-start RMSE={cold:.3f} -> final RMSE={final:.3f}  "
      f"(one-shot local was {baseline_table['one-shot local'][0]:.3f}, "
      f"manifold {baseline_table['manifold'][0]:.3f}, pseudoinv {baseline_table['pseudoinv'][0]:.3f})")
print(f"    on-manifold throughout? geodesic local resid={geo_resid_l:.3f} vs real={real_res_local:.3f} "
      f"(ratio {geo_resid_l/real_res_local:.2f})")

# (b) observation space: reach target + ghost
geo_to_tgt = obs_table["geodesic"][0]; uns_to_tgt = obs_table["unsteered"][0]
geo_ghost  = obs_table["geodesic"][2]
frac_closed = (uns_to_tgt - geo_to_tgt) / uns_to_tgt if uns_to_tgt > 1e-6 else 0.0
print(f"\n(b) OBSERVATION SPACE:")
print(f"    ->target render: unsteered={uns_to_tgt:.3f}  geodesic={geo_to_tgt:.3f}  "
      f"({100*frac_closed:.0f}% of the gap closed)")
print(f"    ghost ratio (geodesic) = {geo_ghost:.2f}  "
      f"(~0 = old object removed, ~1 = phantom remains)")
reached = "REACHES target" if frac_closed > 0.6 else (
    "PARTIALLY moves toward target" if frac_closed > 0.15 else "does NOT reach target")
ghost_v = "ghost RESOLVED" if geo_ghost < 0.3 else (
    "ghost PARTIALLY resolved" if geo_ghost < 0.7 else "GHOST REMAINS")
print(f"    -> {reached}; {ghost_v}")

# (c) geodesic vs one-shot baselines, one line (readout + obs)
print(f"\n(c) GEODESIC vs ONE-SHOT (readout RMSE | ->target render | ghost):")
for n in ["pseudoinv", "manifold", "one-shot local", "geodesic"]:
    rr = baseline_table[n][0] if n in baseline_table else geo_rmse_final
    print(f"    {n:16s}  readout={rr:.3f}  ->target={obs_table[n][0]:.3f}  ghost={obs_table[n][2]:.2f}")

# (d) both-spaces consistency flag
dec_geo = dec_dist_per_obj(roll_hs["geodesic"]).mean()
print(f"\n(d) BOTH-SPACES CHECK: geodesic decoded-dist-to-target={dec_geo:.3f}, "
      f"obs ->target render={geo_to_tgt:.3f}")
if dec_geo < 0.3 and frac_closed < 0.3:
    print("    *** DISAGREEMENT: probe says object reached target but obs barely moved. ***")
elif dec_geo < 0.3 and frac_closed > 0.6:
    print("    AGREEMENT: both readout and obs reached target.")
else:
    print("    partial / mixed — see tables above.")
print("\nPNGs: /tmp/geodesic/{1_convergence,2_obs_space_metrics,3_scans,4_waterfalls}.png")